# 00 環境チェック

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asg1223/rcj-soccer-ml/blob/main/notebooks/00_check_environment.ipynb)

このノートブックが最後まで動けば、教材を進める準備はできています。
上のセルから順に、`Shift + Enter` で実行してください。

- **Colab** の人：最初のセルで教材リポジトリを自動で取ってきます
- **ローカル** の人：先に `pip install -r requirements.txt` を済ませておきます（0-2 を参照）

In [ ]:
# Colab のときだけ、教材リポジトリを取ってきて読み込めるようにする
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q --depth 1 https://github.com/Asg1223/rcj-soccer-ml.git /content/rcj-soccer-ml
    sys.path.insert(0, "/content/rcj-soccer-ml")
print("Colab で実行中" if IN_COLAB else "ローカルで実行中")

## 1. ライブラリが読み込めるか

In [ ]:
import platform
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import rcjml

print("Python    ", platform.python_version())
print("NumPy     ", np.__version__)
print("pandas    ", pd.__version__)
print("matplotlib", matplotlib.__version__)
print("rcjml     ", rcjml.__version__)

## 2. 設定ファイルが読めるか

何も指定しないと、教材用の例 `configs/examples/ring12.yaml`（IR センサ 12 個）を読みます。
自分の機体の設定を使う方法は 0-4 で説明します。

In [ ]:
from rcjml import load_config, describe

cfg = load_config()
print(describe(cfg))

## 3. シミュレータでデータを作れるか

ボールをランダムな場所に置いたときのセンサ値を、シミュレータで 5 個だけ作ってみます。
`s0`, `s1`, … が各センサの値、`angle_deg` がボールの角度（正面が 0°）、`distance_cm` が距離です。

In [ ]:
from rcjml.sim import make_dataset

df = make_dataset(cfg, n=5, seed=1)
df

## 4. グラフが描けるか

ボールが左斜め前（+45°）、40 cm のところにあるときの、センサ値を円グラフ風に描きます。
オレンジの線が本当のボールの方向です。ボールの方向のセンサほど値が大きくなっていれば OK です。

（グラフの文字は、Colab で日本語が文字化けしないように英語にしています）

In [ ]:
from rcjml.sim import simulate

values = simulate(cfg, angle_deg=45, distance_cm=40, rng=np.random.default_rng(0))[0]
theta = np.deg2rad(cfg.ir.angles_deg)
width = 2 * np.pi / cfg.ir.sensor_count * 0.8

fig = plt.figure(figsize=(4.5, 4.5))
ax = fig.add_subplot(projection="polar")
ax.set_theta_zero_location("N")   # 正面（0°）を上に
ax.bar(theta, values, width=width, alpha=0.7)
ax.plot([np.deg2rad(45)] * 2, [0, cfg.ir.max_reading], color="tab:orange", lw=3)
ax.set_ylim(0, cfg.ir.max_reading)
ax.set_title("IR sensor readings (ball at +45 deg, 40 cm)")
plt.show()

## できたら

グラフが表示されれば準備完了です。サイトに戻って、0-3 に進みましょう。

**うまくいかないとき：** エラーの文と、どのセルで止まったかを控えて、0-2 の「つまずきポイント」を見てください。